In [ ]:
import numpy as np                      # Numerical computations
import pandas as pd                     # Data manipulation (DataFrames)
import matplotlib.pyplot as plt         # Plotting and visualization
import os                               # Operating system utilities

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

data_path = os.path.join(path, 'Q1_data.csv')  # Create full path to CSV file
df_Q1 = pd.read_csv(data_path)                 # Read CSV into DataFrame

print(f"Dataset shape: {df_Q1.shape}")         # Show number of rows and columns

In [ ]:
# Task 2: Write your code here:
df_Q1.head()                                   # Display first 5 rows

In [ ]:
# Task 3: Write your code here:
df_Q1.info()                                 # Display dataset information

In [ ]:
# Task 4: Write your code here:
df_Q1.describe()       # Show statistical description using

In [ ]:
# Task 5: Write your code here:
# target distribution
plt.figure(figsize=(10, 8))
plt.hist(df_Q1['Delivery_Time'].dropna(), bins=50, color='pink', edgecolor='black') # dropna() removes rows with missing values.
plt.title('target distribution')
plt.show()

In [ ]:
# Task 1: Write your code here:
cols = ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_Q1[cols].copy()
df_clean = df_clean.drop(columns=['Order_ID'])
df_clean.head()

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df_Q1.isnull().sum() / len(df_Q1))
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,              # Column names
    'Missing_Percentage': missing_percentage.values  # Missing value percentages
})
missing_data = missing_data[
    missing_data['Missing_Percentage'] > 0
].sort_values('Missing_Percentage', ascending=False) # Keep columns with missing values and sort by percentage
print("Missing Data Analysis:")                      # Print title
missing_data.head(10)                                # Show top 10 columns
#----------------------------------------------------

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')
    # Replace missing text values with 'unknown'

# Fill numarical data with mode, mean
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(
 df_clean['Courier_Experience_yrs'].mode()[0] ) # Replace missing Courier_Experience_yrs with the most common value


df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(
 df_clean['Delivery_Time'].mode()[0] ) # Replace missing Delivery_Time with the mean

print("Missing values remaining:", df_clean.isnull().sum().sum()) # Check total remaining missing values


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # no need to scale target
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

def check_target_imbalance(df_clean, target_column):
  print("Target Distribution:")

  df_clean[target_column].hist()
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
#it is not impalanced

In [ ]:
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q


In [ ]:
# Task 1: Write your code here: 1. Split the dataset into features (X) and target (y)
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold #cuz its not impalanced

num_classes = len(np.unique(y_encoded))

# 1. Initialize the machine learning models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)}
# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y[train_index], y[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)


sklearn_models = {
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}
all_results = {}


In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: